In [ ]:
# basic imports
import os
import numpy as np
import pandas as pd
from tess_backml import BackgroundCube
import matplotlib.pyplot as plt
from IPython.display import HTML
from astropy.time import Time
import functools
from scipy.sparse import csr_matrix
from tqdm import tqdm

# increase animation frame limits
import matplotlib
matplotlib.rcParams["animation.embed_limit"] = 2**128

In [ ]:
from glob import glob
import pandas as pd
from tqdm import tqdm

In [ ]:
from astropy.visualization import simple_norm
from matplotlib import animation, axes, colors
from typing import Callable, Optional, Tuple, Union

def plot_img(
    tdx: int,
    extent: Optional[Tuple] = None,
    cbar: bool = True,
    ax: Optional[plt.Axes] = None,
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    cnorm: Optional[colors.Normalize] = None,
    plot_asteroids: bool = False,
    vmag_lim: float = 21.0,
    cumulative: bool = False,
) -> axes.Axes:
    """
    Plots an image with optional scatter points and colorbar.

    Parameters
    ----------
    img : np.ndarray
        The 2D image data to be plotted.
    scol_2d : Optional[np.ndarray], optional
        The column coordinates of scatter points, by default None.
    srow_2d : Optional[np.ndarray], optional
        The row coordinates of scatter points, by default None.
    plot_type : str, optional
        The type of plot to create ('img' or 'scatter'), by default "img".
    extent : Optional[Tuple], optional
        The extent of the image (left, right, bottom, top), by default None.
    cbar : bool, optional
        Whether to display a colorbar, by default True.
    ax : Optional[plt.Axes], optional
        The matplotlib Axes object to plot on, by default None. If None, a new figure and axes are created.
    title : str, optional
        The title of the plot, by default "".
    vmin : Optional[float], optional
        The minimum value for the colormap, by default None.
    vmax : Optional[float], optional
        The maximum value for the colormap, by default None.
    cnorm : Optional[colors.Normalize], optional
        Custom color normalization, by default None.
    bar_label : str, optional
        The label for the colorbar, by default "Flux [e-/s]"."

    Returns
    -------
    matplotlib.axes
    """
    # Initialise ax
    if ax is None:
        _, ax = plt.subplots()

    # if cumulative:
    #     ffi = build_ffi_cum(tdx=tdx)
    # else:
    ffi = build_ffi(tdx=tdx)

    iso = Time(pred_cube["time"][tdx], format="jd", scale="tdb").iso[:-4]
    title = f"CAD {pred_cube['cadenceno'][tdx]} | Time {iso}"

    # Define vmin and vmax
    if cnorm is None and vmin is None and vmax is None:
        vmin, vmax = np.nanpercentile(ffi.ravel(), [3, 97])
    ffi[ffi>=vmax] = 1

    # Plot image, colorbar and marker
    im = ax.imshow(
        ffi,
        origin="lower",
        cmap="Greys",
        vmin=vmin,
        vmax=vmax,
        norm=cnorm,
        extent=extent,
        rasterized=True,
    )
    if cbar:
        plt.colorbar(im, location="right", shrink=0.8, ax=ax).set_label(label="Probability", size=14)

    if plot_asteroids:
        # mask = average_mag.values.ravel() <= vmag_lim
        # # print(tdx, vmag_lim, mask.sum())
        # ax.scatter(
        #     pos_col.values[mask, tdx] - cmin, 
        #     pos_row.values[mask, tdx] - rmin, 
        #     s=50, 
        #     facecolors='none', 
        #     edgecolors='tab:red'
        # )
        ax.scatter(
            pred_pos_col.values[:, tdx],# - cmin, 
            pred_pos_row.values[:, tdx],# - rmin, 
            s=50, 
            facecolors='none', 
            edgecolors='tab:blue'
        )

    ax.set_aspect("equal", "box")
    ax.set_title(title, fontsize=14)

    # ax.set_xlabel("Pixel Column")
    # ax.set_ylabel("Pixel Row")
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False)
    ax.set_xlim(cmin, cmax)
    ax.set_ylim(rmin, rmax)

    return ax

def animate_cube(
    cadenceno: np.ndarray,
    extent: Optional[Tuple] = None,
    interval: int = 200,
    repeat_delay: int = 1000,
    step: int = 1,
    suptitle: str = "",
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    plot_asteroids: bool = False,
    vmag_lim: float = 21.0,
    cumulative: bool = False,
    cbar: bool = True,
) -> animation.FuncAnimation:
    """
    Animates a 3D data cube (time series of 2D images).

    Parameters
    ----------
    cube : np.ndarray
        The 3D data cube to be animated (time, row, column).
    scol_2d : Optional[np.ndarray], optional
        The column coordinates of scatter points, by default None.
    srow_2d : Optional[np.ndarray], optional
        The row coordinates of scatter points, by default None.
    cadenceno : Optional[np.ndarray], optional
        Cadence numbers corresponding to the time axis, by default None.
    time : Optional[np.ndarray], optional
        Time values corresponding to the time axis, by default None.
    plot_type : str, optional
        The type of plot to create ('img' or 'scatter'), by default "img".
    extent : Optional[Tuple], optional
        The extent of the images (left, right, bottom, top), by default None.
    interval : int, optional
        Delay between frames in milliseconds, by default 200.
    repeat_delay : int, optional
        Delay in milliseconds before repeating the animation, by default 1000.
    step : int, optional
        Step size for frame selection, by default 1.
    suptitle : str, optional
        Overall title for the animation figure, by default "".
    bar_label : str, optional
        The label for the colorbar, by default "Flux [e-/s]".

    Returns
    -------
    animation.FuncAnimation
        The matplotlib FuncAnimation object.
    """
    # Initialise figure and set title
    fig, ax = plt.subplots(figsize=(12, 12))
    fig.suptitle(suptitle, y=.998, fontsize=20)
    plt.tight_layout()

    if vmin is None and vmax is None:
        vmin=-10
        vmax=10

    # Plot first image in cube.
    nt = 0

    ax = plot_img(
        nt,
        extent=extent,
        cbar=cbar,
        ax=ax,
        vmin=vmin,
        vmax=vmax,
        plot_asteroids=plot_asteroids,
        vmag_lim=vmag_lim,
        cumulative=False,
    )

    # Define function for animation
    def animate(nt):
        ax.clear()
        _ = plot_img(
            nt,
            extent=extent,
            cbar=cbar,
            ax=ax,
            vmin=vmin,
            vmax=vmax,
            plot_asteroids=plot_asteroids,
            vmag_lim=vmag_lim,
            cumulative=True,
        )

        return ()

    # Prevent second figure from showing up in interactive mode
    plt.close(ax.figure)  # type: ignore

    # Create the animation
    print(cadenceno[::step])
    ani = animation.FuncAnimation(
        fig,
        animate,
        frames=cadenceno[::step],
        interval=interval,
        blit=True,
        repeat_delay=repeat_delay,
        repeat=True,
    )

    return ani

In [ ]:
# define sector/camera/ccd
sector = 3
camera = 1
ccd = 4

In [ ]:
pred_cube = np.load(f"/Users/jimartin/Work/TESS/tess-asteroid-ml/data/pred_cubes/sector{sector:04}/asteroid_{sector}_{camera}_{ccd}_v4.npz")
pred_cube

In [ ]:
rmin, rmax = 0, 2048
cmin, cmax = 45, 2093
btjd0 = 2457000

In [ ]:
pred_cube["diff"].shape

In [ ]:
plt.imshow(pred_cube["diff"], vmin=0, vmax=0.5, origin="lower")
plt.show()

In [ ]:
ROW, COL = np.arange(pred_cube["diff"].shape[0]), np.arange(pred_cube["diff"].shape[1])

@functools.lru_cache(maxsize=128)
def build_ffi(tdx=0):
    
    idx_mask = pred_cube["pred_indices"][0] == tdx
    pred_R = ROW[pred_cube["pred_indices"][1, idx_mask]]
    pred_C = COL[pred_cube["pred_indices"][2, idx_mask]]
    pred_V = pred_cube["pred_vals"][idx_mask]
    
    ffi = csr_matrix((pred_V, (pred_R, pred_C)), shape=pred_cube["diff"].shape)
    ffi = np.array(ffi.todense())
    
    return ffi

def build_ffi_cum(tdx=0, step=5):
    if tdx < step:
        return build_ffi(tdx=0)
    cum_ffi = np.array([build_ffi(tdx=t) for t in range(0,tdx,step)])
    cum_ffi = cum_ffi.sum(axis=0) #/ len(range(0,tdx,2))
    return cum_ffi

In [ ]:
plot_img(654, vmin=0, vmax=0.5);

In [ ]:
fnames = sorted(glob(f"/Users/jimartin/Work/TESS/tess-asteroid-ml/data/pred_asteroid_tracks/candidate_files/track_candidate_TESS_{sector}_{camera}_{ccd}_*.csv"))
print(len(fnames))

times = pred_cube["time"]

pred_pos_row = pd.DataFrame(columns=times)
pred_pos_col = pd.DataFrame(columns=times)

for k, f in tqdm(enumerate(fnames), total=len(fnames)):
    track = pd.read_csv(f, index_col=0)
    tr_row = np.interp(times, track.time + btjd0, track.row, left=np.nan, right=np.nan)
    tr_col = np.interp(times, track.time + btjd0, track.column, left=np.nan, right=np.nan)
    pred_pos_row.loc[k] = tr_row
    pred_pos_col.loc[k] = tr_col
    # break

In [ ]:
pred_pos_row.head()

In [ ]:
for k in range(len(pred_pos_row)):
    plt.scatter(pred_pos_col.iloc[k], pred_pos_row.iloc[k], s=1)
plt.show()

In [ ]:
cadno = np.arange(pred_cube["cadenceno"].shape[0])

total = pred_cube["time"].shape[0]
# total = 200
step = 5
vmag_lim = 19
cadno[1:total][::step]

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(pred_cube["diff"], vmin=0, vmax=0.5, origin="lower",cmap="Grays")
plt.title(f"TESS Sector {sector} Camera {camera} CCD {ccd}\n Time Aggregated NN Prediction", fontsize=15)
plt.xlim(cmin,cmax)
plt.ylim(rmin,rmax)
plt.xlabel("Pixel Column", fontsize=13)
plt.ylabel("Pixel Row", fontsize=13)
plt.show()

In [ ]:
ani = animate_cube(
        cadno[1:total],
        vmin=0,
        vmax=0.4,
        step=step,
        suptitle=f"TESS Sector {sector} Camera {camera} CCD {ccd}\n NN Prediction",
        plot_asteroids=True,
        vmag_lim=vmag_lim,
        cumulative=False,
        cbar=False
        )
HTML(ani.to_jshtml())

In [ ]:
file_name = f"./figures/sector{sector:03}_{camera}-{ccd}_nnpred_ani_t{total:04}_s{step:02}.gif"
ani.save(file_name, writer="pillow")
file_name

In [ ]:
step